# Aula 7 — BERT e HuggingFace (PyTorch)

Disciplina **CCM-109 Deep Learning** — Prof. Ronaldo Prati (UFABC)

**Tópicos:**
1. MLM Demo — preenchendo lacunas com BERT
2. Embeddings de texto — semântica no espaço vetorial
3. Embeddings não-textuais — imagens e tabelas
4. Visualização de atenção BERT
5. Fine-tuning para classificação de sentimento

In [ ]:
# Instalar dependências
!pip install transformers datasets sentence-transformers torch torchvision --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

---
## Parte 1 — MLM Demo: BERT prevê tokens mascarados

O pré-treinamento do BERT usa **Masked Language Modeling (MLM)**:
- 15% dos tokens são mascarados com `[MASK]`
- O modelo deve prever o token original a partir do contexto bidirecional

Vamos explorar o BERT pré-treinado fazendo previsões de lacunas.

In [ ]:
from transformers import pipeline

# Pipeline de fill-mask — usa bert-base-uncased por padrão
fill_mask = pipeline('fill-mask', model='bert-base-uncased', device=0 if torch.cuda.is_available() else -1)

In [ ]:
def show_predictions(text, top_k=5):
    """Mostra as top_k previsões para tokens mascarados."""
    results = fill_mask(text, top_k=top_k)
    print(f'Texto: "{text}"\n')
    print(f'{"Token":20s}  {"Score":>8s}  Frase')
    print('-' * 60)
    for r in results:
        print(f"{r['token_str']:20s}  {r['score']:8.4f}  {r['sequence']}")
    print()

In [ ]:
# Exemplo 1: semântica contextual
show_predictions("The capital of France is [MASK].")

# Exemplo 2: contexto desambigua o sentido
show_predictions("She went to the bank to [MASK] money.")
show_predictions("She sat on the river [MASK] and watched the sunset.")

In [ ]:
# Exemplo 3: conhecimento de mundo
show_predictions("Albert Einstein was born in [MASK].")
show_predictions("The [MASK] is the largest planet in the solar system.")

In [ ]:
# Visualização: distribuição de probabilidade para uma máscara
text = "Python is a popular programming [MASK]."
results = fill_mask(text, top_k=10)

tokens = [r['token_str'] for r in results]
scores = [r['score'] for r in results]

fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(tokens)))
bars = ax.barh(tokens[::-1], scores[::-1], color=colors[::-1])
ax.set_xlabel('Probabilidade')
ax.set_title(f'BERT fill-mask: "{text}"', pad=10)
for bar, score in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=9)
ax.set_xlim(0, max(scores) * 1.25)
plt.tight_layout()
plt.show()

---
## Parte 2 — Embeddings de texto com BERT

BERT gera **embeddings contextualizados**: o mesmo token terá vetores diferentes dependendo do contexto.

Usaremos o `[CLS]` como representação global da sentença e computaremos similaridade coseno entre sentenças.

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name).to(device)
bert.eval()
print('BERT carregado.')

In [ ]:
def get_bert_embedding(texts, pooling='cls'):
    """
    Extrai embeddings de sentenças usando BERT.
    pooling: 'cls' usa [CLS], 'mean' usa média de todos os tokens.
    """
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = bert(**inputs)
    
    last_hidden = outputs.last_hidden_state  # (B, T, 768)
    
    if pooling == 'cls':
        return last_hidden[:, 0, :].cpu().numpy()  # [CLS] token
    else:  # mean pooling
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        return (last_hidden * mask).sum(1).div(mask.sum(1)).cpu().numpy()

In [ ]:
# Grupo de sentenças semanticamente relacionadas e não relacionadas
sentences = [
    # Tecnologia / Programação
    'Python is a programming language used for machine learning.',
    'Deep learning requires large amounts of training data.',
    'Neural networks learn hierarchical representations.',
    # Esporte
    'The soccer team won the championship last night.',
    'Basketball players need excellent coordination and speed.',
    'The marathon runner finished the race in record time.',
    # Culinária
    'The pasta was cooked with olive oil and fresh tomatoes.',
    'Baking bread requires flour, yeast, water, and salt.',
    'The chef prepared a delicious chocolate cake for dessert.',
]
labels = ['Tech']*3 + ['Sport']*3 + ['Food']*3
colors_map = {'Tech': '#3b82f6', 'Sport': '#ef4444', 'Food': '#10b981'}

embs = get_bert_embedding(sentences, pooling='mean')
print(f'Shape dos embeddings: {embs.shape}')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# PCA 2D para visualização
pca = PCA(n_components=2)
embs_2d = pca.fit_transform(embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: PCA ---
ax = axes[0]
for i, (x, y) in enumerate(embs_2d):
    color = colors_map[labels[i]]
    ax.scatter(x, y, color=color, s=120, zorder=3)
    ax.annotate(sentences[i][:35] + '…', (x, y),
                textcoords='offset points', xytext=(8, 4), fontsize=7)

# Legenda manual
for cat, col in colors_map.items():
    ax.scatter([], [], color=col, label=cat, s=80)
ax.legend()
ax.set_title('PCA 2D dos embeddings BERT (mean pooling)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.grid(alpha=0.3)

# --- Plot 2: Matriz de similaridade coseno ---
ax2 = axes[1]
embs_norm = normalize(embs)
sim_matrix = embs_norm @ embs_norm.T

short = [s[:30] + '…' for s in sentences]
im = ax2.imshow(sim_matrix, cmap='RdYlGn', vmin=0.5, vmax=1.0)
ax2.set_xticks(range(len(sentences)))
ax2.set_yticks(range(len(sentences)))
ax2.set_xticklabels(short, rotation=45, ha='right', fontsize=7)
ax2.set_yticklabels(short, fontsize=7)
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax2.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax2)
ax2.set_title('Similaridade coseno entre sentenças')

plt.suptitle('Embeddings BERT: semântica no espaço vetorial', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Parte 3 — Embeddings não-textuais com HuggingFace

O mesmo paradigma de embeddings se aplica além do texto:
- **Sentence Transformers**: embeddings de sentenças otimizados para similaridade
- **CLIP**: embeddings de texto e imagem no mesmo espaço vetorial
- **Dados tabulares**: categorias como embeddings

### 3.1 Sentence Transformers — embeddings de alta qualidade para similaridade

In [ ]:
from sentence_transformers import SentenceTransformer

# Modelo otimizado para similaridade semântica (fine-tunado com triplets/NLI)
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('Sentence-BERT carregado.')

In [ ]:
# Comparação: BERT-CLS vs Sentence-BERT para recuperação semântica
query = "How to train a machine learning model?"
candidates = [
    "Steps to fit a neural network on data.",        # semanticamente próximo
    "The weather was sunny and warm today.",          # não relacionado
    "Training procedures for deep learning systems.", # próximo
    "My dog loves playing fetch in the park.",        # não relacionado
    "Gradient descent optimizes model parameters.",   # muito próximo
]

q_emb_bert  = get_bert_embedding([query], pooling='mean')
c_embs_bert = get_bert_embedding(candidates, pooling='mean')

q_emb_sbert  = sbert.encode([query])
c_embs_sbert = sbert.encode(candidates)

from sklearn.preprocessing import normalize as sk_norm
sim_bert  = (sk_norm(q_emb_bert) @ sk_norm(c_embs_bert).T)[0]
sim_sbert = (sk_norm(q_emb_sbert) @ sk_norm(c_embs_sbert).T)[0]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(candidates))
width = 0.35
ax.bar(x - width/2, sim_bert, width, label='BERT mean pool', color='#3b82f6', alpha=0.8)
ax.bar(x + width/2, sim_sbert, width, label='Sentence-BERT', color='#10b981', alpha=0.8)
labels_c = [c[:35]+'…' for c in candidates]
ax.set_xticks(x)
ax.set_xticklabels(labels_c, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Similaridade coseno com a query')
ax.set_title(f'Query: "{query[:50]}"')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 CLIP — texto e imagem no mesmo espaço vetorial

O modelo CLIP (Radford et al., 2021) aprende embeddings de texto e imagem no mesmo espaço:
- Texto e imagem com conteúdo similar terão vetores próximos
- Permite busca semântica texto→imagem e imagem→texto

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
from io import BytesIO

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('CLIP carregado.')

In [ ]:
# URLs de imagens de exemplo (Creative Commons)
image_urls = {
    'dog':   'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg',
    'cat':   'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Image_created_with_a_mobile_phone.png/320px-Image_created_with_a_mobile_phone.png',
    'pizza': 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg/320px-Eq_it-na_pizza-margherita_sep2005_sml.jpg',
}

images = {}
for name, url in image_urls.items():
    try:
        resp = requests.get(url, timeout=10)
        images[name] = Image.open(BytesIO(resp.content)).convert('RGB')
        print(f'Carregada: {name}')
    except Exception as e:
        print(f'Erro ao carregar {name}: {e}')

In [ ]:
# Textos candidatos para zero-shot classification
text_candidates = [
    'a photo of a dog',
    'a photo of a cat',
    'a photo of a pizza',
    'a photo of a car',
    'a photo of a forest',
]

if images:
    fig, axes = plt.subplots(1, len(images), figsize=(14, 5))
    
    for ax, (img_name, img) in zip(axes, images.items()):
        # Processar imagem e textos
        inputs = clip_proc(
            text=text_candidates,
            images=img,
            return_tensors='pt',
            padding=True
        ).to(device)
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        logits = outputs.logits_per_image[0]  # (num_texts,)
        probs = logits.softmax(dim=0).cpu().numpy()
        
        # Mostrar imagem
        ax.imshow(img)
        ax.axis('off')
        title = f'{img_name.upper()}\n'
        for t, p in sorted(zip(text_candidates, probs), key=lambda x: -x[1])[:3]:
            title += f'{t}: {p:.2%}\n'
        ax.set_title(title, fontsize=8)
    
    plt.suptitle('CLIP: classificação zero-shot texto→imagem', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('Nenhuma imagem carregada. Verifique a conexão com a internet.')

### 3.3 Embeddings tabulares — categorias como vetores

Variáveis categóricas podem ser mapeadas para vetores densos com `nn.Embedding`.
O Transformer então captura interações entre colunas (TabTransformer).

In [ ]:
import torch.nn as nn

# Dataset sintético: previsão de churn de clientes
np.random.seed(42)
N = 200

# Variáveis categóricas
plano     = np.random.choice(['basic', 'standard', 'premium'], N)  # 3 categorias
regiao    = np.random.choice(['N', 'S', 'L', 'O', 'C'], N)        # 5 regiões
pagamento = np.random.choice(['credit', 'debit', 'boleto'], N)    # 3 métodos

# Variáveis numéricas
meses_contrato = np.random.exponential(12, N).clip(1, 60).astype(int)
valor_mensal   = np.random.normal(150, 50, N).clip(50, 300)

# Vocabulários
plano_vocab     = {v: i for i, v in enumerate(['basic', 'standard', 'premium'])}
regiao_vocab    = {v: i for i, v in enumerate(['N', 'S', 'L', 'O', 'C'])}
pagamento_vocab = {v: i for i, v in enumerate(['credit', 'debit', 'boleto'])}

# Converter para índices
plano_idx     = torch.tensor([plano_vocab[p]     for p in plano])
regiao_idx    = torch.tensor([regiao_vocab[r]    for r in regiao])
pagamento_idx = torch.tensor([pagamento_vocab[p] for p in pagamento])

print(f'Dataset: {N} clientes, 3 variáveis categóricas + 2 numéricas')

In [ ]:
# Camadas de embedding para cada variável categórica
EMB_DIM = 8

emb_plano     = nn.Embedding(3, EMB_DIM)
emb_regiao    = nn.Embedding(5, EMB_DIM)
emb_pagamento = nn.Embedding(3, EMB_DIM)

# Obter embeddings
with torch.no_grad():
    e_plano     = emb_plano(plano_idx).numpy()      # (N, 8)
    e_regiao    = emb_regiao(regiao_idx).numpy()    # (N, 8)
    e_pagamento = emb_pagamento(pagamento_idx).numpy()  # (N, 8)

# Concatenar com variáveis numéricas
num_features = np.stack([meses_contrato, valor_mensal], axis=1)  # (N, 2)
X = np.hstack([e_plano, e_regiao, e_pagamento, num_features])     # (N, 26)

print(f'Feature matrix shape: {X.shape}')
print(f'  = 3 embeddings × {EMB_DIM} dim + 2 numéricas')

In [ ]:
# Visualizar embeddings de 'plano' por PCA
plano_emb_pca = PCA(n_components=2).fit_transform(emb_plano.weight.detach().numpy())
plano_labels  = ['basic', 'standard', 'premium']
plano_colors  = ['#ef4444', '#f97316', '#10b981']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (emb_layer, cats, name) in zip(axes, [
    (emb_plano, ['basic', 'standard', 'premium'], 'Plano'),
    (emb_regiao, ['N', 'S', 'L', 'O', 'C'], 'Região'),
    (emb_pagamento, ['credit', 'debit', 'boleto'], 'Pagamento'),
]):
    vecs = emb_layer.weight.detach().numpy()
    pca2 = PCA(n_components=2).fit_transform(vecs)
    colors = plt.cm.Set1(np.linspace(0, 1, len(cats)))
    
    for i, (cat, col) in enumerate(zip(cats, colors)):
        ax.scatter(*pca2[i], color=col, s=200, zorder=3)
        ax.annotate(cat, pca2[i], textcoords='offset points',
                    xytext=(8, 4), fontsize=10, fontweight='bold')
    
    ax.set_title(f'Embeddings — {name}')
    ax.grid(alpha=0.3)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Embeddings tabulares (inicializados aleatoriamente, treino aprenderia posições significativas)',
             fontsize=9, y=1.01)
plt.tight_layout()
plt.show()

---
## Parte 4 — Visualização de atenção BERT

Extrairemos os pesos de atenção de cada cabeça e camada do BERT e criaremos heatmaps para analisar quais tokens "prestam atenção" em quais.

In [ ]:
from transformers import BertModel, BertTokenizer

# BertModel com output_attentions=True
bert_attn = BertModel.from_pretrained('bert-base-uncased',
                                       output_attentions=True).to(device)
bert_attn.eval()
print('BERT com saída de atenção carregado.')

In [ ]:
def get_attention(text, layer=11):
    """Extrai pesos de atenção de uma camada específica.
    
    Returns:
        tokens: lista de tokens (incluindo [CLS] e [SEP])
        attn:   tensor (num_heads, seq_len, seq_len)
    """
    inputs = tokenizer(text, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    with torch.no_grad():
        outputs = bert_attn(**inputs)
    
    # attentions: tuple de 12 tensores (1, num_heads, seq_len, seq_len)
    attn = outputs.attentions[layer][0].cpu().numpy()  # (12, T, T)
    return tokens, attn

In [ ]:
sentence = "The bank can guarantee deposits will eventually cover future tuition costs."
tokens, attn_last = get_attention(sentence, layer=11)

print(f'Tokens: {tokens}')
print(f'Forma dos pesos de atenção (última camada): {attn_last.shape}')  # (12, T, T)

In [ ]:
def plot_attention_heads(tokens, attn, n_heads=6, layer=11):
    """Plota heatmaps de atenção para múltiplas cabeças."""
    n_cols = 3
    n_rows = (n_heads + n_cols - 1) // n_cols
    T = len(tokens)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4.5, n_rows*4))
    axes = axes.flatten()
    
    for h in range(n_heads):
        ax = axes[h]
        im = ax.imshow(attn[h], cmap='Blues', aspect='auto', vmin=0, vmax=attn[h].max())
        ax.set_xticks(range(T))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(T))
        ax.set_yticklabels(tokens, fontsize=8)
        ax.set_title(f'Cabeça {h+1}', fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    for h in range(n_heads, len(axes)):
        axes[h].axis('off')
    
    plt.suptitle(f'Pesos de atenção — Camada {layer+1} (primeiras {n_heads} cabeças)', fontsize=11)
    plt.tight_layout()
    plt.show()

plot_attention_heads(tokens, attn_last, n_heads=6, layer=11)

In [ ]:
# Atenção média entre camadas — comparação layer 1 vs layer 12
_, attn_first = get_attention(sentence, layer=0)
_, attn_last  = get_attention(sentence, layer=11)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (attn, title) in zip(axes, [
    (attn_first.mean(axis=0), 'Camada 1 — média das 12 cabeças'),
    (attn_last.mean(axis=0),  'Camada 12 — média das 12 cabeças'),
]):
    im = ax.imshow(attn, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.suptitle(f'"{sentence}"', fontsize=9, y=1.01)
plt.tight_layout()
plt.show()

print('\nObservação: camadas iniciais tendem a focar tokens adjacentes;')
print('camadas profundas capturam relações semânticas de longa distância.')

---
## Parte 5 — Fine-tuning para classificação de sentimento

Utilizaremos BERT como encoder e adicionaremos uma cabeça linear para classificação de sentimento no dataset **SST-2** (Stanford Sentiment Treebank).

**Estratégia:** full fine-tuning com learning rate baixo para evitar catastrophic forgetting.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

# Carregar SST-2 do HuggingFace Hub
dataset = load_dataset('glue', 'sst2')
print(dataset)

In [ ]:
# Exemplos do dataset
for split in ['train', 'validation']:
    n = len(dataset[split])
    print(f'{split}: {n} exemplos')

print('\nAmostra:')
for ex in dataset['train'].select(range(5)):
    label = 'positivo' if ex['label'] == 1 else 'negativo'
    print(f"  [{label}] {ex['sentence'][:70]}…")

In [ ]:
class SSTDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=128):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(
            item['sentence'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(item['label'], dtype=torch.long)
        }

BATCH_SIZE = 32
MAX_LEN    = 64

# Subsets para demonstração rápida
train_ds = SSTDataset(dataset['train'].shuffle(seed=42).select(range(2000)), tokenizer, MAX_LEN)
val_ds   = SSTDataset(dataset['validation'], tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} exemplos ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} exemplos ({len(val_loader)} batches)')

In [ ]:
# Modelo: BERT + cabeça de classificação linear
NUM_LABELS = 2  # positivo / negativo
EPOCHS     = 3
LR         = 2e-5

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=NUM_LABELS
).to(device)

# Parâmetros da cabeça vs encoder
head_params    = ['classifier']
encoder_params = [p for n, p in model.named_parameters() if not any(hp in n for hp in head_params)]
head_params_l  = [p for n, p in model.named_parameters() if  any(hp in n for hp in head_params)]

# Layer-wise LR: cabeça com 10× mais lr
optimizer = AdamW([
    {'params': encoder_params, 'lr': LR},
    {'params': head_params_l,  'lr': LR * 10},
], weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Total de parâmetros: {sum(p.numel() for p in model.parameters()):,}')
print(f'Total de steps: {total_steps} (warmup: {warmup_steps})')

In [ ]:
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler(enabled=torch.cuda.is_available())

def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        
        optimizer.zero_grad()
        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        scaler.scale(outputs.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        preds = outputs.logits.argmax(dim=-1)
        total_loss    += outputs.loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)
    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        preds = outputs.logits.argmax(dim=-1)
        total_loss    += outputs.loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)
    return total_loss / total, total_correct / total

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, scaler)
    vl_loss, vl_acc = evaluate(model, val_loader)
    
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    
    print(f'Epoch {epoch}/{EPOCHS} | '
          f'train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
          f'val loss {vl_loss:.4f} acc {vl_acc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, history['train_loss'], 'o-', label='Train', color='#3b82f6')
ax1.plot(epochs, history['val_loss'],   's-', label='Val',   color='#ef4444')
ax1.set_xlabel('Época'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['train_acc'], 'o-', label='Train', color='#3b82f6')
ax2.plot(epochs, history['val_acc'],   's-', label='Val',   color='#ef4444')
ax2.set_xlabel('Época'); ax2.set_ylabel('Acurácia')
ax2.set_title('Acurácia'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle('Fine-tuning BERT — SST-2 (sentimento)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Inferência em frases novas
def predict_sentiment(texts):
    model.eval()
    inputs = tokenizer(texts, return_tensors='pt', padding=True,
                       truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs  = logits.softmax(dim=-1).cpu().numpy()
    labels = ['negativo', 'positivo']
    for text, prob in zip(texts, probs):
        pred = labels[prob.argmax()]
        conf = prob.max()
        print(f'[{pred} {conf:.2%}] {text[:70]}')

predict_sentiment([
    "This movie was absolutely fantastic and I loved every minute of it!",
    "The film was a complete waste of time, terribly boring.",
    "It was an okay movie, nothing special but not terrible either.",
    "One of the best performances I have ever seen on screen.",
])

---
## Resumo

| Tópico | O que vimos |
|---|---|
| **MLM** | BERT preenche lacunas usando contexto bidirecional; contexto resolve ambiguidade |
| **Embeddings texto** | Vetores 768-dim, similaridade coseno captura semântica; mean pooling > CLS para similaridade |
| **Sentence-BERT** | Fine-tuning com pares de similaridade produz embeddings melhores para busca |
| **CLIP** | Texto e imagem no mesmo espaço vetorial; zero-shot classification |
| **Embeddings tabulares** | Variáveis categóricas → vetores densos; Transformer captura interações |
| **Atenção BERT** | Camadas iniciais: atenção local; camadas profundas: relações semânticas |
| **Fine-tuning** | BERT + cabeça linear; lr=2e-5, warmup 10%; layer-wise decay melhora estabilidade |